# HMCN-F Focused Hyperparameter Tuning

This notebook runs a focused 6-config grid search to validate whether
the previously tuned `lr` and `weight_decay` values still hold after
switching from `ReduceLROnPlateau` to `CosineAnnealingWarmRestarts`.

**Fixed parameters (from previous tuning):**
- global_dim = 128
- local_dim = 64
- dropout = 0.47
- lambda_viol = 0.1
- beta = 0.5

**Parameters being searched:**
- lr ∈ {1e-3, 5e-4, 1e-4}
- weight_decay ∈ {1e-4, 1e-3}

**Model selection criterion:** Validation Meta ROC AUC (threshold-independent, stable signal)

**Stratification:** MultilabelStratifiedShuffleSplit on Y1 (138 fine labels)

## 1. Install dependencies

In [ ]:
!pip install iterative-stratification -q

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 3. Hierarchy definition (Vassilis's lookup table)

In [ ]:
META_CATEGORIES = {
    'floral':        ['floral','rose','jasmin','lily','muguet','violet','hyacinth',
                      'geranium','lavender','orangeflower','chamomile','hawthorn'],
    'fruity':        ['fruity','apple','apricot','banana','berry','cherry','grape',
                      'grapefruit','lemon','melon','orange','peach','pear','pineapple',
                      'plum','raspberry','strawberry','tropical','black currant','fruit skin'],
    'sweet':         ['sweet','vanilla','caramellic','honey','chocolate','cocoa',
                      'coconut','creamy','buttery','milky','dairy'],
    'woody':         ['woody','cedar','sandalwood','pine','vetiver','terpenic',
                      'balsamic','cortex'],
    'green':         ['green','grassy','herbal','leafy','hay','tea','fresh',
                      'cucumber','vegetable','weedy'],
    'spicy':         ['spicy','cinnamon','clove','warm','pungent','sharp',
                      'cooling','mint','camphoreous'],
    'animal_musk':   ['animal','musk','leathery','fishy','sweaty','meaty',
                      'beefy','musty'],
    'earthy':        ['earthy','mushroom','nutty','hazelnut','roasted','coffee',
                      'tobacco','smoky','popcorn'],
    'citrus':        ['citrus','bergamot','ozone','clean','soapy'],
    'chemical':      ['solvent','ethereal','metallic','medicinal','phenolic',
                      'sulfurous','gassy','burnt','oily'],
    'gourmand':      ['almond','malty','rummy','brandy','cognac','winey','cooked',
                      'potato','savory','celery','tomato','radish','onion','garlic',
                      'cabbage','cheesy'],
    'powdery_amber': ['amber','powdery','anisic','coumarinic','orris','waxy',
                      'aldehydic','ketonic','lactonic'],
}

## 4. Data loading and splitting

Stratification is done on **Y1 (138 fine labels)** — more principled than Y2 since
we directly protect the proportions of what the model is learning.
All 138 fine labels have ≥31 positives so the algorithm handles them cleanly.

In [ ]:
def load_data(csv_path='hmcn_dataset.csv'):
    df = pd.read_csv(csv_path)

    fine_cols = [c for c in df.columns if c.startswith('fine_')]
    meta_cols = [c for c in df.columns if c.startswith('meta_')]
    feat_cols = [c for c in df.columns if c not in fine_cols + meta_cols + ['SMILES']]

    # Drop zero-variance features — they carry no information
    stds = df[feat_cols].std()
    feat_cols = stds[stds > 0].index.tolist()

    X  = df[feat_cols].values.astype(np.float32)
    Y1 = df[fine_cols].values.astype(np.float32)   # 138 fine labels
    Y2 = df[meta_cols].values.astype(np.float32)   # 12 metacategories

    fine_names = [c.replace('fine_', '') for c in fine_cols]
    meta_names = [c.replace('meta_', '') for c in meta_cols]

    return X, Y1, Y2, fine_names, meta_names


def split_and_scale(X, Y1, Y2, seed=42):
    # Train+Val / Test split — stratified on Y1
    msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    trainval_idx, test_idx = next(msss.split(X, Y1))

    # Train / Val split — stratified on Y1
    msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.1/0.8, random_state=seed)
    train_idx, val_idx = next(msss2.split(X[trainval_idx], Y1[trainval_idx]))

    X_train = X[trainval_idx][train_idx]
    X_val   = X[trainval_idx][val_idx]
    X_test  = X[test_idx]

    Y1_train = Y1[trainval_idx][train_idx]
    Y1_val   = Y1[trainval_idx][val_idx]
    Y1_test  = Y1[test_idx]

    Y2_train = Y2[trainval_idx][train_idx]
    Y2_val   = Y2[trainval_idx][val_idx]
    Y2_test  = Y2[test_idx]

    # StandardScaler fitted on train only — avoids data leakage
    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    return (X_train, Y1_train, Y2_train,
            X_val,   Y1_val,   Y2_val,
            X_test,  Y1_test,  Y2_test)


def build_violation_pairs(fine_names, meta_names):
    """
    Returns list of (fine_idx, meta_idx) pairs from Vassilis's lookup table.
    Used in the hierarchical violation penalty: P(fine) should never exceed P(meta).
    """
    fine_idx = {name: i for i, name in enumerate(fine_names)}
    meta_idx = {name: i for i, name in enumerate(meta_names)}
    pairs = []
    for meta, members in META_CATEGORIES.items():
        for member in members:
            if member in fine_idx and meta in meta_idx:
                pairs.append((fine_idx[member], meta_idx[meta]))
    return pairs


# Load and split
X, Y1, Y2, fine_names, meta_names = load_data('hmcn_dataset.csv')
violation_pairs = build_violation_pairs(fine_names, meta_names)

(
    X_train, Y1_train, Y2_train,
    X_val,   Y1_val,   Y2_val,
    X_test,  Y1_test,  Y2_test
) = split_and_scale(X, Y1, Y2)

print(f'Train : {len(X_train)} molecules')
print(f'Val   : {len(X_val)} molecules')
print(f'Test  : {len(X_test)} molecules')
print(f'Features        : {X_train.shape[1]}')
print(f'Fine labels     : {Y1.shape[1]}')
print(f'Meta labels     : {Y2.shape[1]}')
print(f'Hierarchy pairs : {len(violation_pairs)}')

## 5. DataLoaders

In [ ]:
def make_loader(X, Y1, Y2, batch_size, shuffle):
    dataset = TensorDataset(
        torch.tensor(X,  dtype=torch.float32),
        torch.tensor(Y1, dtype=torch.float32),
        torch.tensor(Y2, dtype=torch.float32)
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


train_loader = make_loader(X_train, Y1_train, Y2_train, batch_size=32, shuffle=True)
val_loader   = make_loader(X_val,   Y1_val,   Y2_val,   batch_size=128, shuffle=False)
test_loader  = make_loader(X_test,  Y1_test,  Y2_test,  batch_size=128, shuffle=False)

## 6. Model definition

HMCN-F with 2-level hierarchy:
- **Level 1** (LocalBlock): fine labels (138)
- **Level 2** (LocalBlock): metacategories (12)
- **Global output**: all 150 labels simultaneously

Information flows: x → A_G^0 → (level1 → A_G^1, P_L1) → (level2 → A_G^2, P_L2) → P_G

Final prediction: P_F = β * [P_L1 | P_L2] + (1-β) * P_G

In [ ]:
class LocalBlock(nn.Module):
    """
    One hierarchical level of HMCN-F.

    Takes:
      x   : original input (input reuse — each level can see raw features directly)
      A_G : global hidden state from previous level

    Returns:
      A_G_next : updated global hidden state
      P_L      : local predictions for this level (probabilities)
    """
    def __init__(self, input_dim, global_dim, local_dim, n_labels, dropout):
        super().__init__()

        # Global flow: (A_G ⊕ x) → next A_G
        self.global_fc = nn.Sequential(
            nn.Linear(global_dim + input_dim, global_dim),
            nn.BatchNorm1d(global_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Transition: A_G → local hidden A_L
        self.transition = nn.Sequential(
            nn.Linear(global_dim, local_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Local output: A_L → probabilities for this level's labels
        self.output = nn.Linear(local_dim, n_labels)

    def forward(self, x, A_G):
        A_G_next = self.global_fc(torch.cat([A_G, x], dim=1))
        A_L      = self.transition(A_G_next)
        P_L      = torch.sigmoid(self.output(A_L))
        return A_G_next, P_L


class HMCNF(nn.Module):
    def __init__(self, input_dim, n_fine, n_meta,
                 global_dim, local_dim, dropout, beta):
        super().__init__()

        self.beta    = beta
        self.n_total = n_fine + n_meta

        # Initial projection: x → A_G^0
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, global_dim),
            nn.BatchNorm1d(global_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Level 1: fine labels (138)
        self.level1 = LocalBlock(input_dim, global_dim, local_dim, n_fine, dropout)

        # Level 2: metacategories (12)
        self.level2 = LocalBlock(input_dim, global_dim, local_dim, n_meta, dropout)

        # Global output: A_G^2 → all 150 labels at once
        self.global_output = nn.Linear(global_dim, self.n_total)

    def forward(self, x):
        A_G       = self.input_proj(x)
        A_G, P_L1 = self.level1(x, A_G)
        A_G, P_L2 = self.level2(x, A_G)
        P_G       = torch.sigmoid(self.global_output(A_G))
        P_local   = torch.cat([P_L1, P_L2], dim=1)
        P_F       = self.beta * P_local + (1 - self.beta) * P_G
        return P_F, P_L1, P_L2, P_G

## 7. Loss function

Total loss = Local loss + Global loss + λ * Hierarchical violation penalty

- **Local loss**: BCE on fine predictions + BCE on meta predictions separately
- **Global loss**: BCE on all 150 labels at once from the global output head
- **Violation penalty**: penalizes P(child) > P(parent) for each hierarchy pair

In [ ]:
def binary_cross_entropy(P, Y, eps=1e-7):
    P = torch.clamp(P, eps, 1 - eps)
    return -torch.mean(Y * torch.log(P) + (1 - Y) * torch.log(1 - P))


def hierarchical_violation_penalty(P_L1, P_L2, pairs):
    """
    For each (child, parent) pair: penalty = mean( max(0, P_child - P_parent)^2 )
    Averaged over all pairs so lambda_viol stays scale-invariant.
    """
    total = torch.tensor(0.0, device=P_L1.device)
    for fine_idx, meta_idx in pairs:
        violation = torch.clamp(P_L1[:, fine_idx] - P_L2[:, meta_idx], min=0.0)
        total = total + torch.mean(violation ** 2)
    return total / max(len(pairs), 1)


def hmcn_loss(P_F, P_L1, P_L2, P_G, Y1, Y2, pairs, lambda_viol):
    Y_global = torch.cat([Y1, Y2], dim=1)

    local_loss     = binary_cross_entropy(P_L1, Y1) + binary_cross_entropy(P_L2, Y2)
    global_loss    = binary_cross_entropy(P_G, Y_global)
    violation_loss = hierarchical_violation_penalty(P_L1, P_L2, pairs)

    return local_loss + global_loss + lambda_viol * violation_loss

## 8. Training and evaluation helpers

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, pairs, lambda_viol, device):
    model.train()
    total_loss = 0.0

    for batch_idx, (X_batch, Y1_batch, Y2_batch) in enumerate(loader):
        X_batch  = X_batch.to(device)
        Y1_batch = Y1_batch.to(device)
        Y2_batch = Y2_batch.to(device)

        optimizer.zero_grad()
        P_F, P_L1, P_L2, P_G = model(X_batch)
        loss = hmcn_loss(P_F, P_L1, P_L2, P_G, Y1_batch, Y2_batch, pairs, lambda_viol)
        loss.backward()

        # Gradient clipping — prevents exploding gradients
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # Warm restart scheduler step — called per batch for CosineAnnealingWarmRestarts
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def collect_predictions(model, loader, device):
    """Run model on loader, collect raw probability outputs."""
    model.eval()
    fine_probs, fine_true = [], []
    meta_probs, meta_true = [], []

    for X_batch, Y1_batch, Y2_batch in loader:
        _, P_L1, P_L2, _ = model(X_batch.to(device))
        fine_probs.append(P_L1.cpu().numpy())
        fine_true.append(Y1_batch.numpy())
        meta_probs.append(P_L2.cpu().numpy())
        meta_true.append(Y2_batch.numpy())

    return (
        np.vstack(fine_probs), np.vstack(fine_true),
        np.vstack(meta_probs), np.vstack(meta_true)
    )


def compute_macro_roc_auc(probs, targets):
    """Macro-average ROC AUC across all labels that have at least one positive."""
    aucs = []
    for i in range(targets.shape[1]):
        if targets[:, i].sum() > 0:
            aucs.append(roc_auc_score(targets[:, i], probs[:, i]))
    return np.mean(aucs) if aucs else 0.0


def compute_macro_pr_auc(probs, targets):
    """Macro-average PR AUC across all labels that have at least one positive."""
    pr_aucs = []
    for i in range(targets.shape[1]):
        if targets[:, i].sum() > 0:
            pr_aucs.append(average_precision_score(targets[:, i], probs[:, i]))
    return np.mean(pr_aucs) if pr_aucs else 0.0

## 9. Focused grid search

6 configs varying only `lr` and `weight_decay`.
All other parameters fixed at values from previous tuning.

**Scheduler**: CosineAnnealingWarmRestarts
- T_0 = 50 epochs (initial restart period)
- T_mult = 2 (each restart period doubles: 50 → 100 → 200)
- eta_min = 1e-5 (minimum learning rate at bottom of cosine curve)

In [ ]:
# Fixed hyperparameters
GLOBAL_DIM   = 128
LOCAL_DIM    = 64
DROPOUT      = 0.47
LAMBDA_VIOL  = 0.1
BETA         = 0.5
BATCH_SIZE   = 32
EPOCHS       = 150      # enough for 3 warm restart cycles (50+100)
PATIENCE     = 40       # generous patience to allow restarts to help

# Grid — only lr and weight_decay vary
CONFIGS = [
    {'lr': 1e-3, 'weight_decay': 1e-4},
    {'lr': 1e-3, 'weight_decay': 1e-3},
    {'lr': 5e-4, 'weight_decay': 1e-4},
    {'lr': 5e-4, 'weight_decay': 1e-3},
    {'lr': 1e-4, 'weight_decay': 1e-4},
    {'lr': 1e-4, 'weight_decay': 1e-3},
]

print(f'Running {len(CONFIGS)} configs...')
print(f'Fixed: global_dim={GLOBAL_DIM}, local_dim={LOCAL_DIM}, '
      f'dropout={DROPOUT}, lambda_viol={LAMBDA_VIOL}, beta={BETA}')
print()

results = []
best_val_auc  = 0.0
best_config   = None

header = f"{'#':>3}  {'lr':>6}  {'wd':>6}  {'best_ep':>8}  {'val_AUC':>9}  {'val_PRAUC':>10}  {'tst_AUC':>9}  {'tst_PRAUC':>10}"
print(header)
print('-' * len(header))

for config_idx, config in enumerate(CONFIGS):

    # Build model
    model = HMCNF(
        input_dim  = X_train.shape[1],
        n_fine     = Y1_train.shape[1],
        n_meta     = Y2_train.shape[1],
        global_dim = GLOBAL_DIM,
        local_dim  = LOCAL_DIM,
        dropout    = DROPOUT,
        beta       = BETA
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr           = config['lr'],
        weight_decay = config['weight_decay']
    )

    # CosineAnnealingWarmRestarts:
    # T_0 = steps per first cycle = epochs_per_cycle * batches_per_epoch
    steps_per_epoch = len(train_loader)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0    = 50 * steps_per_epoch,   # restart every 50 epochs
        T_mult = 2,
        eta_min= 1e-5
    )

    best_val_auc_config = 0.0
    best_state          = None
    patience_counter    = 0
    best_epoch          = 0

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(
            model, train_loader, optimizer, scheduler,
            violation_pairs, LAMBDA_VIOL, device
        )
        fp_v, ft_v, mp_v, mt_v = collect_predictions(model, val_loader, device)
        val_meta_auc = compute_macro_roc_auc(mp_v, mt_v)

        if val_meta_auc > best_val_auc_config:
            best_val_auc_config = val_meta_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            best_epoch = epoch
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

    # Evaluate best checkpoint on test set
    model.load_state_dict(best_state)
    model.to(device)

    fp_v, ft_v, mp_v, mt_v = collect_predictions(model, val_loader, device)
    fp_t, ft_t, mp_t, mt_t = collect_predictions(model, test_loader, device)

    val_meta_pr_auc  = compute_macro_pr_auc(mp_v, mt_v)
    test_meta_auc    = compute_macro_roc_auc(mp_t, mt_t)
    test_meta_pr_auc = compute_macro_pr_auc(mp_t, mt_t)

    marker = ' ◄ best' if best_val_auc_config > best_val_auc else ''
    if best_val_auc_config > best_val_auc:
        best_val_auc = best_val_auc_config
        best_config  = config

    print(
        f"{config_idx+1:>3}  {config['lr']:>6.0e}  {config['weight_decay']:>6.0e}  "
        f"{best_epoch:>8d}  {best_val_auc_config:>9.4f}  {val_meta_pr_auc:>10.4f}  "
        f"{test_meta_auc:>9.4f}  {test_meta_pr_auc:>10.4f}{marker}"
    )

    results.append({
        'lr'            : config['lr'],
        'weight_decay'  : config['weight_decay'],
        'best_epoch'    : best_epoch,
        'val_meta_auc'  : best_val_auc_config,
        'val_meta_prauc': val_meta_pr_auc,
        'tst_meta_auc'  : test_meta_auc,
        'tst_meta_prauc': test_meta_pr_auc,
    })

print()
print(f'Best config: lr={best_config["lr"]:.0e}, weight_decay={best_config["weight_decay"]:.0e}')
print(f'Best val Meta ROC AUC: {best_val_auc:.4f}')

## 10. Save tuning results

In [ ]:
results_df = pd.DataFrame(results).sort_values('val_meta_auc', ascending=False)
results_df.to_csv('hmcn_focused_tune_results.csv', index=False)

print('Full results (sorted by val Meta ROC AUC):')
print(results_df.to_string(index=False))
print()
print('Saved → hmcn_focused_tune_results.csv')
print()
print('Use the best config (top row) in hmcn_final.ipynb')